# Statistische Analyse

Dieses Notebook wertet `output/statistics.csv` aus und erlaubt:
- Ausschluss bestimmter `Vehicles` und `Scenarios`
- Auswahl beliebiger `Player` ü den Vergleich
- Vergleich der Noten-Häufigkeit ü variable Disziplinen (z. B. `Median`, `P90`, `P95`)


In [2]:
import pandas as pd
import plotly.express as px
from pathlib import Path
import os
pd.set_option('display.max_columns', 50)
from datetime import datetime



## Chapter 1 Vorverarbeitung filtert nach Datum etc.

In [1]:
# Ab welchem Datum soll ausgewertet werden?
AnfangsDatum="20260219"
EndDatum="20551231"
output_file="legacy_boostclga.csv"

In [3]:
data=pd.DataFrame()
output_dir = Path(os.getcwd()) / "output"
if not output_dir.exists():
    print(f"Output-Verzeichnis nicht gefunden: {output_dir}")
#    return 1
Datumfilter="20260219"

for csv_path in sorted(output_dir.glob("*.csv")):
    try:
        date = datetime.strptime(csv_path.name[:8], "%Y%m%d")
        if date >= datetime.strptime(AnfangsDatum, "%Y%m%d") and date <= datetime.strptime(EndDatum, "%Y%m%d"):
            df=pd.read_csv(csv_path, sep=";")
            df['filename']=csv_path.name
            data=pd.concat([data, df])
            #print(csv_path.name)
    except ValueError:
        #print('Error')
        pass
data.drop(columns=['timestamp','datetime_local','ocr_confidence','source_file'], inplace=True)
out_file = output_dir / output_file
data.to_csv(out_file, sep=";", index=False)


## Chapter 2: Detailed Comparison
### Get the Data

In [ ]:
# Pfad zur CSV-Datei
CSV_PATH = 'output/statistics.csv'

df = pd.read_csv(CSV_PATH, encoding='utf-8', encoding_errors='replace', sep=";")

# Häufige Encoding-Artefakte (z. B. ÃƒÆ’Ã‚Â¼) in String-Spalten bereinigen
def _fix_mojibake(val):
    if not isinstance(val, str):
        return val
    mapping = {
        'ÃƒÆ’Ã‚Â¤': 'ä', 'ÃƒÆ’Ã‚Â¶': 'ÃƒÂ¶', 'ÃƒÆ’Ã‚Â¼': 'ÃƒÂ¼', 'ÃƒÆ’Ã¢â‚¬Å¾': 'Ãƒâ€ž', 'ÃƒÆ’Ã¢â‚¬â€œ': 'Ãƒâ€“', 'ÃƒÆ’Ã…â€œ': 'ÃƒÅ“', 'ÃƒÆ’Ã…Â¸': 'ÃƒÅ¸'
    }
    for bad, good in mapping.items():
        val = val.replace(bad, good)
    return val

#for c in df.select_dtypes(include='object').columns:
#    df[c] = df[c].map(_fix_mojibake)
display(df.head(3))
print(f'Anzahl Datensätze: {len(df)}')
print('Spalten:', ', '.join(df.columns))
#print(f'Players: {df["Player"].value_counts()}')


,Dateiname,Mittelwert,Mittelwert_Note,Median,Median_Note,P90,P90_Note,P95,P95_Note,hervorragend,sehr gut,gut,ausreichend,ungenügend
0,20260129_HOME_WLAN_Verstärker_BOOSTCLGA_LOADYO...,20.9,gut,19.0,sehr gut,24.0,gut,29.0,gut,0.0,58.5,39.0,2.0,0.5
1,20260129_HOME_WLAN_Verstärker_BOOSTNOPRIO_LOAD...,16.9,sehr gut,16.0,sehr gut,20.0,gut,23.0,gut,0.0,88.0,12.0,0.0,0.0
2,20260129_HOME_WLAN_Verstärker_BOOSTNOPRIO_LOAD...,23.1,gut,22.0,gut,30.0,gut,34.0,gut,0.0,30.2,67.8,2.0,0.0


Anzahl Datensätze: 95
Spalten: Dateiname, Mittelwert, Mittelwert_Note, Median, Median_Note, P90, P90_Note, P95, P95_Note, hervorragend, sehr gut, gut, ausreichend, ungenügend


KeyError: 'Player'

## Set the Filters

In [ ]:
# Konfiguration
# Diese Listen kannst du direkt anpassen.

# 1) Werte, die NICHT berücksichtigt werden sollen
exclude_vehicles = ["Extender", 'LAN']
exclude_scenarios = ["STEAM"]


# 2) Player, die verglichen werden sollen
selected_players = ['Optima_Prio_CLGA', 'Magenta_Home']

# 3) Disziplinen, deren _Note-Spalten verglichen werden
selected_disciplines = ['Median', 'P90', 'P95']

# 4) Testläufe die verglichen werden sollen
selected_runs = ['20260219','20260210']
#selected_runs = ['20260219']
valid_disciplines = ['Mittelwert', 'Median', 'P90', 'P95']
invalid = [d for d in selected_disciplines if d not in valid_disciplines]
if invalid:
    raise ValueError(f'Ungültige Disziplin(en): {invalid}. Erlaubt: {valid_disciplines}')

for d in selected_disciplines:
    note_col = f'{d}_Note'
    if note_col not in df.columns:
        raise KeyError(f'Spalte fehlt: {note_col}')



## Apply the Filters

In [ ]:
# Filter anwenden
df_f = df.copy()

if exclude_vehicles:
    df_f = df_f[~df_f['Vehicles'].isin(exclude_vehicles)]
if exclude_scenarios:
    df_f = df_f[~df_f['Scenarios'].isin(exclude_scenarios)]
if selected_players:
    df_f = df_f[df_f['Player'].isin(selected_players)]
if selected_runs:
    df_0=pd.DataFrame()
    for k in selected_runs:
        df_0=pd.concat([df_0, df_f[df_f['Dateiname'].str.contains(k)]])
        #df_f= df_f[df_f['Dateiname'].str.contains(k)]
    df_f = df_0
    

print(f'Gefilterte Datensätze: {len(df_f)}')
display(df_f[['Player', 'Vehicles', 'Scenarios']].head(16))
print(f'{df_f["Vehicles"].value_counts()},\n{df_f["Scenarios"].value_counts()}')

## Calculate Noten-Häufigkeit pro Player und Disziplin berechnen

In [ ]:
# Noten-Häufigkeit pro Player und Disziplin berechnen
grade_order = ['hervorragend', 'sehr gut', 'gut', 'ausreichend', 'ungenügend']

parts = []
for d in selected_disciplines:
    note_col = f'{d}_Note'
    g = (
        df_f.groupby(['Player', note_col], dropna=False)
        .size()
        .reset_index(name='Anzahl')
        .rename(columns={note_col: 'Note'})
    )
    g['Disziplin'] = d
    parts.append(g)

freq = pd.concat(parts, ignore_index=True) if parts else pd.DataFrame(columns=['Player', 'Note', 'Anzahl', 'Disziplin'])
freq['Note'] = pd.Categorical(freq['Note'], categories=grade_order, ordered=True)
freq = freq.sort_values(['Disziplin', 'Player', 'Note']).reset_index(drop=True)

display(freq)


## Visualization: isualisierung: Haeufigkeit je Note, getrennt nach Disziplin

In [ ]:
# Visualisierung: Haeufigkeit je Note, getrennt nach Disziplin (Plotly)
if freq.empty:
    print('Keine Daten nach Filterung vorhanden.')
else:
    p = freq.copy()
    p['Note'] = p['Note'].astype(str)

    fig = px.bar(
        p,
        x='Note',
        y='Anzahl',
        color='Player',
        facet_col='Disziplin',
        barmode='group',
        category_orders={'Note': grade_order, 'Disziplin': selected_disciplines},
        title='Noten-Haeufigkeit je Disziplin'
    )

    fig.update_xaxes(categoryorder='array', categoryarray=grade_order, tickangle=30)
    fig.update_yaxes(title_text='Haeufigkeit')
    fig.for_each_annotation(lambda a: a.update(text=a.text.replace('Disziplin=', 'Disziplin: ')))
    fig.update_layout(legend_title_text='Player')
    fig.show()


## Bester Player je Diziplin (Median, P90, P95, ...)

In [ ]:
# Optional: "Wer hat besser abgeschlossen?" ÃƒÂ¼ber Noten-Score je Disziplin
# Niedriger Score ist besser.
grade_score = {'hervorragend': 1, 'sehr gut': 2, 'gut': 3, 'ausreichend': 4, 'ungenÃƒÂ¼gend': 5}

ranking_rows = []
for d in selected_disciplines:
    note_col = f'{d}_Note'
    tmp = df_f[['Player', note_col]].copy().rename(columns={note_col: 'Note'})
    tmp['Score'] = tmp['Note'].map(grade_score)
    r = tmp.groupby('Player', as_index=False)['Score'].mean().sort_values('Score')
    r['Disziplin'] = d
    ranking_rows.append(r)

ranking = pd.concat(ranking_rows, ignore_index=True) if ranking_rows else pd.DataFrame(columns=['Player', 'Score', 'Disziplin'])
ranking = ranking[['Disziplin', 'Player', 'Score']].sort_values(['Disziplin', 'Score'])

display(ranking)

if not ranking.empty:
    winners = ranking.groupby('Disziplin', as_index=False).first()
    print('Beste Player je Disziplin (nach ÃƒËœ-Noten-Score):')
    display(winners)


## Direkter Szenario-Vergleich (kleinerer Zahlenwert gewinnt)

In [ ]:
# Direkter Szenario-Vergleich (kleinerer Zahlenwert gewinnt)
if len(selected_players) != 2:
    raise ValueError('Fuer den direkten Vergleich bitte genau 2 Player in selected_players setzen.')

p1, p2 = selected_players
vergleich_rows = []

for d in selected_disciplines:
    metric_col = d
    if metric_col not in df_f.columns:
        raise KeyError(f'Spalte fehlt fuer Disziplin {d}: {metric_col}')

    m = (
        df_f[df_f['Player'].isin([p1, p2])]
        .groupby(['Scenarios', 'Player'], as_index=False)[metric_col]
        .median()
    )

    pivot = m.pivot(index='Scenarios', columns='Player', values=metric_col).reset_index()
    if p1 not in pivot.columns or p2 not in pivot.columns:
        continue

    pivot = pivot.dropna(subset=[p1, p2]).copy()
    if pivot.empty:
        continue

    pivot['Disziplin'] = d
    pivot['Gewinner'] = pivot.apply(
        lambda r: p1 if r[p1] < r[p2] else (p2 if r[p2] < r[p1] else 'Unentschieden'),
        axis=1
    )
    pivot['Differenz_ms'] = (pivot[p1] - pivot[p2]).abs()

    vergleich_rows.append(
        pivot[['Disziplin', 'Scenarios', p1, p2, 'Gewinner', 'Differenz_ms']]
        .rename(columns={p1: f'{p1}_ms', p2: f'{p2}_ms'})
    )

vergleich = pd.concat(vergleich_rows, ignore_index=True) if vergleich_rows else pd.DataFrame()

if vergleich.empty:
    print('Keine vergleichbaren Szenario-Daten fuer die beiden Player gefunden.')
else:
    display(vergleich.sort_values(['Disziplin', 'Scenarios']).reset_index(drop=True))

    print('Beispiel-Ausgabe je Szenario:')
    for _, r in vergleich.sort_values(['Disziplin', 'Scenarios']).iterrows():
        print(f"{r['Disziplin']} {p1}: {r[f'{p1}_ms']:.2f} ms")
        print(f"{r['Disziplin']} {p2}: {r[f'{p2}_ms']:.2f} ms")
        print(f"{r['Gewinner']} hat gewonnen.")
        print('-' * 40)

    win_freq = (
        vergleich[vergleich['Gewinner'] != 'Unentschieden']
        .groupby(['Disziplin', 'Gewinner'])
        .size()
        .reset_index(name='Siege')
        .sort_values(['Disziplin', 'Siege'], ascending=[True, False])
    )

    print('Haeufigkeit der Siege je Player und Disziplin:')
    display(win_freq)

    win_total = (
        vergleich[vergleich['Gewinner'] != 'Unentschieden']['Gewinner']
        .value_counts()
        .rename_axis('Player')
        .reset_index(name='Siege_gesamt')
    )

    print('Gesamte Siege ueber alle Disziplinen/Szenarien:')
    display(win_total)
